# LoRA: Low-Rank Adaptation

LoRA是一种参数高效的微调方法，通过低秩矩阵适配大模型。

## 核心概念

**公式：**
$$h = W_0 x + \Delta W x = W_0 x + BA x$$

其中：
- $W_0 \in \mathbb{R}^{d \times k}$ 是冻结的预训练权重
- $B \in \mathbb{R}^{d \times r}$，$A \in \mathbb{R}^{r \times k}$ 是可训练的低秩矩阵
- $r \ll \min(d, k)$ 是秩，决定参数量

**参数量：**
- 全量微调：$d \times k$
- LoRA：$r \times (d + k)$
- 当 $r \ll d, k$ 时，参数量大幅减少

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

## 1. LoRA层实现

In [ ]:
class LoRALayer:
    """LoRA层实现"""
    
    def __init__(self, in_features, out_features, rank=8, alpha=16, pretrained_weight=None):
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        
        # 预训练权重（冻结）
        if pretrained_weight is not None:
            self.W = pretrained_weight
        else:
            self.W = np.random.randn(out_features, in_features) / np.sqrt(in_features)
        
        # LoRA低秩矩阵
        self.A = np.random.randn(rank, in_features) / np.sqrt(in_features)
        self.B = np.zeros((out_features, rank))  # 初始化为0
        
        # 缩放
        self.scaling = alpha / rank
    
    def forward(self, x):
        """前向传播: h = Wx + α/r·BAx"""
        # 原始输出
        output = np.dot(x, self.W.T)
        
        # LoRA输出
        lora_output = np.dot(np.dot(x, self.A.T), self.B.T) * self.scaling
        
        return output + lora_output
    
    def merge_weights(self):
        """合并权重用于推理优化"""
        delta_W = np.dot(self.B, self.A) * self.scaling
        return self.W + delta_W
    
    def get_num_trainable_params(self):
        """计算可训练参数量"""
        lora_params = self.rank * self.in_features + self.out_features * self.rank
        total_params = self.out_features * self.in_features + lora_params
        return lora_params, total_params

# 测试
lora = LoRALayer(in_features=512, out_features=512, rank=8, alpha=16)

trainable, total = lora.get_num_trainable_params()
print(f"原始参数量: {512 * 512:,}")
print(f"LoRA参数量: {trainable:,}")
print(f"压缩比: {(512*512)/trainable:.1f}x")
print(f"参数减少: {(1 - trainable/(512*512))*100:.2f}%")

## 2. 前向传播验证

In [ ]:
# 创建输入
x = np.random.randn(4, 10, 512)  # (batch, seq_len, features)

# LoRA前向传播
output = lora.forward(x)

print(f"输入形状: {x.shape}")
print(f"输出形状: {output.shape}")

# 分解计算
W_out = np.dot(x, lora.W.T)
lora_out = np.dot(np.dot(x, lora.A.T), lora.B.T) * lora.scaling

print(f"\n原始分支(Wx): {W_out.shape}")
print(f"LoRA分支(BAx): {lora_out.shape}")
print(f"\n初始时LoRA输出全为0（因为B初始化为0）: {np.allclose(lora_out, 0)}")

## 3. 权重合并验证

In [ ]:
# 模拟训练后的B（非零）
lora.B = np.random.randn(512, 8) * 0.01

# 分别计算
output_separate = lora.forward(x)

# 合并后计算
W_merged = lora.merge_weights()
output_merged = np.dot(x, W_merged.T)

# 验证等价性
diff = np.max(np.abs(output_separate - output_merged))
print(f"分离计算 vs 合并计算的最大差异: {diff:.10f}")
print(f"\n推理优化：")
print(f"  合并前: 需要2次矩阵乘法 (Wx + BAx)")
print(f"  合并后: 只需1次矩阵乘法 (W_merged·x)")
print(f"  推理速度无额外开销！")

## 4. 不同Rank的参数量对比

In [ ]:
# 参数设置
d = 512
ranks = [1, 2, 4, 8, 16, 32, 64, 128]

full_params = d * d
lora_params_list = [r * d + d * r for r in ranks]
ratios = [lp / full_params * 100 for lp in lora_params_list]

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 参数量对比
ax1.plot(ranks, [full_params] * len(ranks), 'r--', label='Full Fine-tune', linewidth=2)
ax1.plot(ranks, lora_params_list, 'b-o', label='LoRA', linewidth=2)
ax1.set_xlabel('Rank (r)')
ax1.set_ylabel('Number of Parameters')
ax1.set_title('Parameter Count vs Rank')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 参数占比
ax2.bar(range(len(ranks)), ratios, color='steelblue')
ax2.set_xlabel('Rank')
ax2.set_ylabel('Percentage of Full Parameters (%)')
ax2.set_title('LoRA Parameters as % of Full Fine-tune')
ax2.set_xticks(range(len(ranks)))
ax2.set_xticklabels(ranks)
ax2.grid(axis='y', alpha=0.3)

# 添加数值标签
for i, (r, ratio) in enumerate(zip(ranks, ratios)):
    ax2.text(i, ratio + 0.5, f'{ratio:.1f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print("参数量对比表:")
print(f"{'Rank':<8} {'LoRA参数':<15} {'占比':<10} {'压缩比':<10}")
print("-" * 50)
for r, lp, ratio in zip(ranks, lora_params_list, ratios):
    compression = full_params / lp
    print(f"{r:<8} {lp:<15,} {ratio:<10.2f}% {compression:<10.1f}x")

## 5. 多头注意力中应用LoRA

In [ ]:
class MultiHeadAttentionWithLoRA:
    """带LoRA的多头注意力"""
    
    def __init__(self, embed_dim, num_heads, rank=8, alpha=16, use_lora_on=['q', 'v']):
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.use_lora_on = use_lora_on
        
        # 创建Q, K, V, O投影
        # 模拟预训练权重
        pretrained_wq = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        pretrained_wk = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        pretrained_wv = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        pretrained_wo = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        
        # 根据配置决定是否使用LoRA
        self.W_q = LoRALayer(embed_dim, embed_dim, rank, alpha, pretrained_wq) if 'q' in use_lora_on else pretrained_wq
        self.W_k = LoRALayer(embed_dim, embed_dim, rank, alpha, pretrained_wk) if 'k' in use_lora_on else pretrained_wk
        self.W_v = LoRALayer(embed_dim, embed_dim, rank, alpha, pretrained_wv) if 'v' in use_lora_on else pretrained_wv
        self.W_o = LoRALayer(embed_dim, embed_dim, rank, alpha, pretrained_wo) if 'o' in use_lora_on else pretrained_wo
    
    def get_num_trainable_params(self):
        trainable = 0
        total = 0
        
        for layer in [self.W_q, self.W_k, self.W_v, self.W_o]:
            if isinstance(layer, LoRALayer):
                t, tot = layer.get_num_trainable_params()
                trainable += t
            else:
                tot = self.embed_dim * self.embed_dim
            total += tot
        
        return trainable, total

# 不同LoRA配置对比
configs = [
    ('无LoRA（全量微调）', []),
    ('仅Q', ['q']),
    ('Q和V（常用）', ['q', 'v']),
    ('Q,K,V', ['q', 'k', 'v']),
    ('全部（Q,K,V,O）', ['q', 'k', 'v', 'o'])
]

embed_dim = 768
num_heads = 12
rank = 8

print("多头注意力中的LoRA配置对比（embed_dim=768, rank=8）\n")
print(f"{'配置':<20} {'可训练参数':<15} {'总参数':<15} {'占比':<10}")
print("-" * 65)

for name, lora_on in configs:
    if not lora_on:
        # 全量微调
        trainable = 4 * embed_dim * embed_dim
        total = trainable
    else:
        mha = MultiHeadAttentionWithLoRA(embed_dim, num_heads, rank, use_lora_on=lora_on)
        trainable, total = mha.get_num_trainable_params()
    
    ratio = trainable / (4 * embed_dim * embed_dim) * 100
    print(f"{name:<20} {trainable:<15,} {total:<15,} {ratio:<10.2f}%")

## 6. BERT-base规模的参数对比

In [ ]:
# BERT-base配置
d_model = 768
num_layers = 12
d_ffn = d_model * 4  # FFN中间层维度
rank = 8

# 每层的参数量
# Attention: 4 * d * d (Q,K,V,O)
# FFN: d * d_ffn + d_ffn * d
attn_params_per_layer = 4 * d_model * d_model
ffn_params_per_layer = d_model * d_ffn + d_ffn * d_model
params_per_layer = attn_params_per_layer + ffn_params_per_layer

# 全量微调
total_full = num_layers * params_per_layer

# LoRA微调（仅Attention的Q和V）
lora_params_per_attn = 2 * (rank * d_model + d_model * rank)
total_lora = num_layers * lora_params_per_attn

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 参数量对比
categories = ['Full Fine-tune', 'LoRA (rank=8)']
values = [total_full, total_lora]
colors = ['#e74c3c', '#3498db']

bars = ax1.bar(categories, values, color=colors)
ax1.set_ylabel('Number of Parameters')
ax1.set_title('BERT-base: Full Fine-tune vs LoRA')
ax1.set_yscale('log')
ax1.grid(axis='y', alpha=0.3)

# 添加数值标签
for bar, val in zip(bars, values):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:,}',
            ha='center', va='bottom')

# 不同rank的对比
ranks_test = [1, 2, 4, 8, 16, 32, 64]
lora_params_by_rank = [num_layers * 2 * (r * d_model + d_model * r) for r in ranks_test]
ratios_by_rank = [lp / total_full * 100 for lp in lora_params_by_rank]

ax2.plot(ranks_test, ratios_by_rank, 'o-', linewidth=2, markersize=8)
ax2.set_xlabel('Rank')
ax2.set_ylabel('Percentage of Full Parameters (%)')
ax2.set_title('LoRA Parameters vs Rank (BERT-base)')
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log', base=2)

plt.tight_layout()
plt.show()

print("BERT-base LoRA微调分析:")
print(f"\n模型配置:")
print(f"  隐藏维度: {d_model}")
print(f"  层数: {num_layers}")
print(f"  FFN维度: {d_ffn}")

print(f"\n参数量:")
print(f"  全量微调: {total_full:,}")
print(f"  LoRA (rank={rank}, 仅Q/V): {total_lora:,}")

print(f"\n效率:")
print(f"  参数减少: {(1 - total_lora/total_full)*100:.2f}%")
print(f"  压缩比: {total_full/total_lora:.0f}x")
print(f"  显存节省: 约{(1 - total_lora/total_full)*100:.0f}%（梯度+优化器状态）")

## 7. LoRA的秩分解可视化

In [ ]:
# 创建一个小例子来可视化
d_in, d_out, r = 8, 6, 2

# 原始权重
W = np.random.randn(d_out, d_in)

# LoRA矩阵
B = np.random.randn(d_out, r)
A = np.random.randn(r, d_in)

# 低秩更新
delta_W = np.dot(B, A)

# 可视化
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# W
im1 = axes[0, 0].imshow(W, cmap='RdBu_r', aspect='auto')
axes[0, 0].set_title(f'W ({d_out}×{d_in})\nPretrained Weight (Frozen)')
axes[0, 0].set_xlabel('Input Features')
axes[0, 0].set_ylabel('Output Features')
plt.colorbar(im1, ax=axes[0, 0])

# B
im2 = axes[0, 1].imshow(B, cmap='RdBu_r', aspect='auto')
axes[0, 1].set_title(f'B ({d_out}×{r})\nLoRA Matrix B (Trainable)')
axes[0, 1].set_xlabel('Rank')
axes[0, 1].set_ylabel('Output Features')
plt.colorbar(im2, ax=axes[0, 1])

# A
im3 = axes[0, 2].imshow(A, cmap='RdBu_r', aspect='auto')
axes[0, 2].set_title(f'A ({r}×{d_in})\nLoRA Matrix A (Trainable)')
axes[0, 2].set_xlabel('Input Features')
axes[0, 2].set_ylabel('Rank')
plt.colorbar(im3, ax=axes[0, 2])

# ΔW = BA
im4 = axes[1, 0].imshow(delta_W, cmap='RdBu_r', aspect='auto')
axes[1, 0].set_title(f'ΔW = BA ({d_out}×{d_in})\nLow-Rank Update')
axes[1, 0].set_xlabel('Input Features')
axes[1, 0].set_ylabel('Output Features')
plt.colorbar(im4, ax=axes[1, 0])

# W + ΔW
im5 = axes[1, 1].imshow(W + delta_W, cmap='RdBu_r', aspect='auto')
axes[1, 1].set_title(f'W + ΔW ({d_out}×{d_in})\nMerged Weight')
axes[1, 1].set_xlabel('Input Features')
axes[1, 1].set_ylabel('Output Features')
plt.colorbar(im5, ax=axes[1, 1])

# 参数量对比
axes[1, 2].axis('off')
param_text = f"""
参数量分析:

原始权重 W:
  {d_out} × {d_in} = {d_out * d_in} 参数
  (冻结，不训练)

LoRA矩阵:
  B: {d_out} × {r} = {d_out * r}
  A: {r} × {d_in} = {r * d_in}
  总计: {d_out * r + r * d_in} 参数
  (可训练)

压缩比:
  {(d_out * d_in) / (d_out * r + r * d_in):.1f}x

参数占比:
  {(d_out * r + r * d_in) / (d_out * d_in) * 100:.1f}%
"""
axes[1, 2].text(0.1, 0.5, param_text, fontsize=12, family='monospace',
               verticalalignment='center')

plt.tight_layout()
plt.show()

## 8. 实际应用场景

In [ ]:
# 模拟多任务LoRA
class MultiTaskLoRA:
    """多任务LoRA系统"""
    
    def __init__(self, in_features, out_features, rank=8, alpha=16):
        # 共享的预训练权重
        self.W = np.random.randn(out_features, in_features) / np.sqrt(in_features)
        
        # 为每个任务创建独立的LoRA模块
        self.task_loras = {}
        
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
    
    def add_task(self, task_name):
        """为新任务添加LoRA模块"""
        A = np.random.randn(self.rank, self.in_features) / np.sqrt(self.in_features)
        B = np.zeros((self.out_features, self.rank))
        self.task_loras[task_name] = {'A': A, 'B': B}
    
    def forward(self, x, task_name):
        """针对特定任务的前向传播"""
        # 基础输出（共享）
        output = np.dot(x, self.W.T)
        
        # 任务特定的LoRA
        if task_name in self.task_loras:
            lora = self.task_loras[task_name]
            scaling = self.alpha / self.rank
            lora_out = np.dot(np.dot(x, lora['A'].T), lora['B'].T) * scaling
            output = output + lora_out
        
        return output

# 演示
multi_lora = MultiTaskLoRA(512, 512, rank=8)

# 添加多个任务
tasks = ['情感分析', '命名实体识别', '问答', '摘要生成']
for task in tasks:
    multi_lora.add_task(task)

print("多任务LoRA系统:")
print(f"\n共享预训练权重: {512 * 512:,} 参数")
print(f"\n任务数: {len(tasks)}")
lora_per_task = 8 * 512 + 512 * 8
print(f"每任务LoRA参数: {lora_per_task:,}")
print(f"总LoRA参数: {lora_per_task * len(tasks):,}")

print(f"\n优势:")
print(f"  ✓ 只需存储{len(tasks)}个小的LoRA模块")
print(f"  ✓ 任务切换只需替换LoRA权重")
print(f"  ✓ 共享预训练知识，保持泛化能力")

# 测试不同任务
x_test = np.random.randn(1, 512)
print(f"\n测试输入: {x_test.shape}")
for task in tasks:
    out = multi_lora.forward(x_test, task)
    print(f"  {task}: 输出形状 {out.shape}")

## 9. 总结

### LoRA的关键优势

1. **参数高效**：通常减少99%以上的可训练参数
2. **显存友好**：在消费级GPU上微调大模型
3. **推理无开销**：合并权重后速度与原模型相同
4. **多任务灵活**：轻松切换不同任务的LoRA模块

### Rank选择建议

- **rank=1-4**: 极度参数高效，适合简单任务
- **rank=8**: 最常用，平衡效果和效率（论文推荐）
- **rank=16-32**: 复杂任务，接近全量微调效果
- **rank>64**: 很少使用，可能失去LoRA的优势

### 实际应用

- **GPT-3微调**: 减少99.9%参数
- **Stable Diffusion**: 风格迁移、人物定制
- **LLaMA系列**: Alpaca、Vicuna等
- **多任务学习**: 每任务独立LoRA模块

### 最佳实践

1. 通常只在Attention的Q和V投影上应用LoRA
2. 使用 α/r 作为缩放因子，α通常设为16或32
3. 初始化B为0，A使用正态分布
4. 推理时合并权重以消除额外开销